# 02 · YOLOv8 Product Detection
**Owner: P1 (Vision)** · Smart Shelf Market Basket Analysis

1. Download the **Indian Grocery Object Detection** dataset (Roboflow, YOLOv8 format)
2. Explore it (images per split, instances per class)
3. Train YOLOv8 on **whatever device you're on** - Colab GPU, NVIDIA laptop, or Apple Silicon
4. Evaluate → `results/yolo_<tag>.json` (overall + per-class P / R / mAP) + plots
5. **Experiment 1:** public-only vs public + own shelf photos *(once shelf photos are labelled)*

**Training across devices:** every run is saved to the shared `DATA_DIR/runs/`, tagged with the machine
name. To continue a run on a different machine, set `START_WEIGHTS` to the previous run's `last.pt`.
On Colab, turn on the GPU first: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# ============================================================
# SETUP - identical in every notebook. Works on Colab, Mac, Windows.
# ============================================================
import os, sys, subprocess
from pathlib import Path

# Colab only: where to get the code from.
#   - GitHub (recommended): set with  python tools/set_repo_url.py <url>
#   - "" : the product-identification-correlation folder is in Google Drive at MyDrive/product-identification-correlation
REPO_URL = "https://github.com/behlah-rgb/product-identification-correlation.git"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if REPO_URL:
        repo = Path("/content/product-identification-correlation")
        url = REPO_URL
        try:  # private repo: add a GitHub token as Colab secret GITHUB_TOKEN
            from google.colab import userdata
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        if repo.exists():
            subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
        else:
            subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
    else:
        repo = Path("/content/drive/MyDrive/product-identification-correlation")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
else:
    repo = Path.cwd()
    while not (repo / "src").exists() and repo != repo.parent:
        repo = repo.parent

sys.path.insert(0, str(repo))
os.chdir(repo)

from src import env
env.summary()

In [ ]:
import json, yaml
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from src import detection, demo_data
from src.env import DATASETS_DIR, RUNS_DIR, RESULTS_DIR, get_device, device_name
from src.viz import show

# ---- Dataset config (from the Roboflow page's "Download Dataset -> YOLOv8 -> show download code") ----
RF_WORKSPACE = "agentsk47"
RF_PROJECT   = "indian-grocery-object-detection-mfsnx"
RF_VERSION   = 1          # <- check the latest version number on the Roboflow page
DATASET      = "indian_grocery"

DEMO = False              # True = tiny synthetic dataset, just to test the pipeline

## 1. Get the dataset

In [ ]:
if DEMO:
    DATASET = "demo_products"
    dataset_dir = DATASETS_DIR / DATASET
    if not dataset_dir.exists():
        demo_data.make_demo_detection_dataset(dataset_dir)
    print("DEMO MODE - synthetic dataset")
else:
    dataset_dir = DATASETS_DIR / DATASET
    if not (dataset_dir / "data.yaml").exists():
        detection.download_roboflow(RF_WORKSPACE, RF_PROJECT, RF_VERSION, DATASET)

data_yaml = detection.write_local_yaml(dataset_dir)
CLASS_NAMES = detection.read_names(dataset_dir)
print("Dataset:", dataset_dir)
print(f"{len(CLASS_NAMES)} classes:", CLASS_NAMES)

## 2. Explore the dataset

In [ ]:
rows = []
for split in ["train", "valid", "test"]:
    lbl_dir = dataset_dir / split / "labels"
    if not lbl_dir.exists():
        continue
    files = list(lbl_dir.glob("*.txt"))
    for f in files:
        for line in f.read_text().split("\n"):
            if line.strip():
                rows.append({"split": split, "class": CLASS_NAMES[int(line.split()[0])]})
    print(f"{split:6s}: {len(list((dataset_dir / split / 'images').iterdir())):4d} images")

inst = pd.DataFrame(rows)
counts = inst.groupby(["class", "split"]).size().unstack(fill_value=0)
counts.plot.barh(stacked=True, figsize=(9, 0.4 * len(CLASS_NAMES) + 1.5))
plt.xlabel("Instances (boxes)"); plt.title("Instances per class"); plt.tight_layout()
plt.savefig(RESULTS_DIR / "dataset_class_distribution.png", dpi=150); plt.show()
counts

In [ ]:
# A few training images with their ground-truth boxes
samples = sorted((dataset_dir / "train" / "images").iterdir())[:4]
vis = []
for p in samples:
    im = cv2.imread(str(p)); h, w = im.shape[:2]
    lbl = dataset_dir / "train" / "labels" / (p.stem + ".txt")
    for line in (lbl.read_text().split("\n") if lbl.exists() else []):
        if not line.strip():
            continue
        c, xc, yc, bw, bh = map(float, line.split()[:5])
        x1, y1, x2, y2 = int((xc - bw/2) * w), int((yc - bh/2) * h), int((xc + bw/2) * w), int((yc + bh/2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 0, 255), 3)
        cv2.putText(im, CLASS_NAMES[int(c)], (x1, max(y1 - 8, 20)), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    vis.append(im)
show(vis, [p.name[:25] for p in samples], size=4)

## 3. Train
| Machine | Expected time for 50 epochs (~400 images, yolov8n) |
|---|---|
| Colab T4 | ~10-15 min |
| RTX 3050 laptop | ~10-20 min |
| M5 MacBook Air (mps) | ~20-40 min |
| CPU only | 1-2 h - avoid |

`START_WEIGHTS = "yolov8n.pt"` starts fresh from COCO-pretrained weights.
To continue someone else's run on your machine, use `detection.latest_run_weights("public_only")`.

In [ ]:
RUN_NAME      = "public_only"      # experiment name - becomes the run folder prefix
MODEL         = "yolov8n.pt"       # yolov8s.pt = more accurate, ~2x slower
START_WEIGHTS = MODEL              # or: detection.latest_run_weights("public_only")
EPOCHS        = 50 if not DEMO else 5
IMGSZ         = 640 if not DEMO else 320

print(f"Training on {device_name()} | start={START_WEIGHTS} | epochs={EPOCHS}")
run_dir = detection.train(data_yaml, RUN_NAME, weights=START_WEIGHTS, epochs=EPOCHS, imgsz=IMGSZ)
best = run_dir / "weights" / "best.pt"

In [ ]:
# If a Colab session disconnected mid-training, resume it exactly (same machine/paths only):
# run_dir = detection.resume(RUNS_DIR / "public_only_colab-0")

## 4. Evaluate

In [ ]:
result = detection.evaluate(best, data_yaml, tag=RUN_NAME, imgsz=IMGSZ)
print(f"\nmAP@50 = {result['mAP50']:.3f} | mAP@50-95 = {result['mAP50_95']:.3f} | "
      f"P = {result['precision']:.3f} | R = {result['recall']:.3f}")
pd.DataFrame(result["per_class"]).T.round(3).sort_values("mAP50", ascending=False)

In [ ]:
# Training curves
m = pd.read_csv(run_dir / "results.csv"); m.columns = [c.strip() for c in m.columns]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for c in ["train/box_loss", "train/cls_loss", "val/box_loss", "val/cls_loss"]:
    if c in m: ax[0].plot(m["epoch"], m[c], label=c)
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
for c in ["metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/precision(B)", "metrics/recall(B)"]:
    if c in m: ax[1].plot(m["epoch"], m[c], label=c.split("/")[1])
ax[1].set_title("Validation metrics"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR / f"training_curves_{RUN_NAME}.png", dpi=150); plt.show()

In [ ]:
# Confusion matrix + PR curve produced by Ultralytics
from pathlib import Path
plots = Path(result["plots_dir"])
imgs = [cv2.imread(str(plots / f)) for f in ["confusion_matrix_normalized.png", "BoxPR_curve.png"] if (plots / f).exists()]
if imgs:
    show(imgs, ["Normalised confusion matrix", "Precision-Recall curve"][:len(imgs)], size=7)

In [ ]:
# Predictions on test images
from ultralytics import YOLO
model = YOLO(str(best))
test_dir = dataset_dir / ("test" if (dataset_dir / "test").exists() else "valid") / "images"
preds = model.predict(sorted(test_dir.iterdir())[:4], imgsz=IMGSZ, conf=0.35, device=get_device(), verbose=False)
show([p.plot() for p in preds], [f"{len(p.boxes)} detections" for p in preds], size=4)

## 5. Experiment 1 - public-only vs public + own shelf photos
Once P4 has labelled ~150 shelf photos in Roboflow (**same class names** as the public dataset), set
`SHELF_*` below. Both models are then evaluated on the **shelf test split** - the question is how much
in-domain photos help.

In [ ]:
SHELF_WORKSPACE = ""       # your Roboflow workspace
SHELF_PROJECT   = ""       # your shelf-photo project
SHELF_VERSION   = 1
SHELF_DATASET   = "shelf_photos"

shelf_dir = DATASETS_DIR / SHELF_DATASET
if SHELF_PROJECT and not (shelf_dir / "data.yaml").exists():
    detection.download_roboflow(SHELF_WORKSPACE, SHELF_PROJECT, SHELF_VERSION, SHELF_DATASET)

if not (shelf_dir / "data.yaml").exists():
    print("No shelf dataset yet - skip this section for now.")
else:
    merged_dir = detection.merge_datasets(dataset_dir, shelf_dir, "public_plus_shelf")
    merged_yaml = detection.write_local_yaml(merged_dir)
    shelf_yaml = detection.write_local_yaml(shelf_dir, names=detection.read_names(merged_dir))

    run2 = detection.train(merged_yaml, "public_plus_shelf", weights=MODEL, epochs=EPOCHS, imgsz=IMGSZ)
    r_pub = detection.evaluate(best, shelf_yaml, tag="exp1_public_only_on_shelf", imgsz=IMGSZ)
    r_mix = detection.evaluate(run2 / "weights" / "best.pt", shelf_yaml, tag="exp1_public_plus_shelf_on_shelf", imgsz=IMGSZ)
    exp1 = pd.DataFrame([{"train data": "public only", **{k: r_pub[k] for k in ["mAP50", "mAP50_95", "precision", "recall"]}},
                         {"train data": "public + shelf", **{k: r_mix[k] for k in ["mAP50", "mAP50_95", "precision", "recall"]}}])
    exp1.to_csv(RESULTS_DIR / "exp1_domain_gap.csv", index=False)
    display(exp1.round(3))

**For the paper (P1):** dataset table (images/split, classes), training setup (model, epochs, imgsz, batch,
device), overall + per-class metrics table, training curves, confusion matrix, and Experiment 1 table.
All numbers are in `DATA_DIR/results/`.